In [10]:
!pip install -q langchain-groq langchain-core langchain-community pypdf

In [13]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY')

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "Resume_Screener_Assignment"

In [14]:
!mkdir -p prompts chains

In [18]:
%%writefile prompts/templates.py
from langchain_core.prompts import ChatPromptTemplate

SCREENER_SYSTEM_PROMPT = """
You are a Technical Recruiter expert. Evaluate the Resume text against the Job Description (JD).

TASK:
1. Extract: Technical skills, Tools, and Years of Experience.
2. Matching: Compare the Resume content strictly against JD requirements.
3. Scoring: Assign a score (0-100) based on relevance.
4. Explanation: Provide a brief justification for the score.

CONSTRAINTS:
- DO NOT assume skills. If a skill isn't in the text, it doesn't exist.
- Output MUST be valid JSON.

{format_instructions}
"""

user_template = "Job Description:\n{jd}\n\nResume Content:\n{resume}"

def get_screener_prompt(parser):
    return ChatPromptTemplate.from_messages([
        ("system", SCREENER_SYSTEM_PROMPT),
        ("user", user_template)
    ]).partial(format_instructions=parser.get_format_instructions())

Writing prompts/templates.py


In [19]:
%%writefile chains/screener.py
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
# Fixed the import path here
try:
    from langchain_core.pydantic_v1 import BaseModel, Field
except ImportError:
    from pydantic import BaseModel, Field
from typing import List
from prompts.templates import get_screener_prompt

# Define JSON structure for the Bonus requirement
class ScreeningResult(BaseModel):
    extracted_skills: List[str] = Field(description="List of skills found in resume")
    experience_years: float = Field(description="Total years of relevant experience")
    fit_score: int = Field(description="Score from 0 to 100")
    explanation: str = Field(description="Reasoning for the score")

# Initialize Parser and LLM
parser = JsonOutputParser(pydantic_object=ScreeningResult)
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)

# Build LCEL Chain
screener_chain = get_screener_prompt(parser) | llm | parser

Writing chains/screener.py


In [ ]:
from google.colab import files
from pypdf import PdfReader
import io
import json
from chains.screener import screener_chain

def extract_pdf_text():
    uploaded = files.upload()
    if not uploaded: return None
    name = list(uploaded.keys())[0]
    reader = PdfReader(io.BytesIO(uploaded[name]))
    return "".join([page.extract_text() for page in reader.pages])

jd_text = """
Role: Junior AI/ML Engineer
Technical Requirements:
- Strong proficiency in Python and FastAPI.
- Hands-on experience with LangChain and Groq/OpenAI APIs.
- Experience building RAG (Retrieval-Augmented Generation) pipelines.
- Knowledge of Vector Databases (ChromaDB/Pinecone).
Education: Pursuing or completed B.E. in CS/AI/ML.
"""

print("\n" + "="*60)
print("🚀 INITIATING AI RESUME SCREENING PIPELINE 🚀")
print("="*60)

candidate_types = ["Strong", "Average", "Weak"]
all_results = []

for c_type in candidate_types:
    print(f"\n[ACTION] Please upload the resume PDF for the {c_type.upper()} candidate:")
    resume_text = extract_pdf_text()

    if resume_text:
        print("⚙️ Processing document through LangChain pipeline...")
        response = screener_chain.invoke(
            {"jd": jd_text, "resume": resume_text},
            config={"tags": [c_type, "Colab_Run"]}
        )
        all_results.append({"type": c_type, "data": response})
        print(f"✅ {c_type} Candidate Evaluated! (Score: {response.get('fit_score', 0)}/100)")
    else:
        print(f"❌ Upload failed or cancelled. Skipped {c_type} candidate.")

print("\n\n" + "★"*60)
print("🏆 FINAL RECRUITER SCREENING REPORT 🏆")
print("★"*60)

for res in all_results:
    c_type = res['type'].upper()
    data = res['data']
    score = data.get('fit_score', 0)

    # Dynamic status based on score
    if score >= 80:
        status = "🟢 HIGHLY RECOMMENDED"
    elif score >= 50:
        status = "🟡 CONSIDER FOR INTERVIEW"
    else:
        status = "🔴 REJECT"

    print(f"\nCandidate Profile : {c_type}")
    print(f"Match Status      : {status}")
    print(f"Fit Score         : {score}/100")
    print(f"Experience Found  : {data.get('experience_years', 'N/A')} years")

    # Safely handle the skills list
    skills = data.get('extracted_skills', [])
    if isinstance(skills, list):
        print(f"Extracted Skills  : {', '.join(skills)}")
    else:
        print(f"Extracted Skills  : {skills}")

    print(f"AI Reasoning      : {data.get('explanation', 'No explanation provided.')}")
    print("-" * 60)

print("\n✅ Pipeline execution complete. Check LangSmith for tracing details.")


🚀 INITIATING AI RESUME SCREENING PIPELINE 🚀

[ACTION] Please upload the resume PDF for the STRONG candidate:


Saving Resume.pdf to Resume (1).pdf
⚙️ Processing document through LangChain pipeline...
✅ Strong Candidate Evaluated! (Score: 40/100)

[ACTION] Please upload the resume PDF for the AVERAGE candidate:
